In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
import pickle
from sklearn.pipeline import Pipeline

In [2]:
import mlflow

mlflow.set_tracking_uri('http://localhost:5004')
mlflow.set_experiment('nyc_greentaxi_2021_tripdata5004_experiment') 

<Experiment: artifact_location='s3://mlflow-ride-duration21-prediction-artifact-store/1', creation_time=1753513197320, experiment_id='1', last_update_time=1753513197320, lifecycle_stage='active', name='nyc_greentaxi_2021_tripdata5004_experiment', tags={}>

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime)
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds()/60)
    df = df[((df['duration'] > 1) & (df['duration'] <= 60))]
    
    categorical = ['PULocationID','DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [4]:
df_train = read_dataframe('green_tripdata/green_tripdata_2021-01.parquet')
df_val = read_dataframe('green_tripdata/green_tripdata_2021-02.parquet')

In [5]:
categorical = ['PU_DO']
numerical = ['trip_distance']
features = categorical + numerical

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

y_train = df_train['duration'].values
y_val = df_val['duration'].values

In [6]:
import xgboost as xgb
from xgboost import XGBRegressor

In [7]:
import mlflow.xgboost
mlflow.xgboost.autolog(disable=True)

In [8]:
dv = DictVectorizer()
from math import sqrt

In [12]:
pipeline = Pipeline([
    ('vectorizer', dv),
    ('xgb', XGBRegressor(
        n_estimators=50,
        max_depth=30,
        learning_rate=0.0959,
        reg_alpha=0.0181,
        reg_lambda=0.0117,
        min_child_weight=1.0606,
        objective='reg:squarederror',
        seed=42,
        verbosity=1
    ))
])

In [13]:
with mlflow.start_run():
    mlflow.set_tag("engineer", "richkinwe")
    mlflow.set_tag("model", "xgboost")
    mlflow.log_param("train_data_path", "green_tripdata/green_tripdata_2021-01.parquet")
    mlflow.log_param("val_data_path", "green_tripdata/green_tripdata_2021-02.parquet")

    # Fit pipeline with early stopping
    pipeline.fit(
        train_dicts,
        y_train
    )

    pipeline.fit(train_dicts, y_train)

    # Predict and evaluate
    y_pred = pipeline.predict(val_dicts)
    rmse = sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    # Save DictVectorizer
    dv = pipeline.named_steps['vectorizer']
    with open('models.preprocessor.b', 'wb') as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact('models.preprocessor.b', artifact_path='preprocessor')

    # Log model
    mlflow.sklearn.log_model(pipeline, artifact_path="model_pipeline")



2025/07/26 16:15:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run glamorous-pug-970 at: http://localhost:5004/#/experiments/1/runs/79d08684e9e441dfbc066847af4de142
🧪 View experiment at: http://localhost:5004/#/experiments/1
